# NEXTCART — Auditoría de supuestos (S1–S5), entrenamiento y evaluación

Contenido:

1. **Configuración y conexión** al dataset Instacart local (`orders.csv`,
   `order_products__prior.csv`, `order_products__train.csv`, `products.csv`,
   `aisles.csv`, `departments.csv`).
2. **Auditoría de los cinco supuestos**: S1 integridad de esquema; S2 historial
   secuencial; S3 densidad de cestas y cola larga; S4 señal de coocurrencia y
   recompra; S5 evaluación sin fuga de información (leakage).
3. **Entrenamiento** del baseline explicable (frecuencia global + coocurrencia
   dentro de cestas con soporte, confianza y lift).
4. **Evaluación offline** contra las cestas objetivo del split oficial
   (Precision@K, Recall@K, HitRate@K, MAP@K, cobertura, repeat/explore,
   latencia).

> ⚠️ **Nota honesta:** las métricas de este notebook son **offline** (predicción
> de cesta en el split prior/train de Instacart). El dataset no contiene
> impresiones, exposición ni aceptación de usuarios, por lo que el objetivo
> comercial de **≥ 12 % de aceptación NO se valida aquí**: requiere un A/B real
> de dos semanas en producción.

Requisitos: Python ≥ 3.11, `duckdb`, `polars`, `numpy`, `matplotlib`.


## 0. Librerías

Un solo bloque de imports: del estándar (rutas, JSON, tiempos, hash), DuckDB
como motor SQL de lectura por consultas (no materializa el dataset completo en
RAM), Polars y NumPy para datos vectorizados, y Matplotlib para figuras
persistentes.


In [5]:
from __future__ import annotations

import hashlib
import json
import platform
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import duckdb
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import polars as pl

print(f"Python {platform.python_version()} · duckdb {duckdb.__version__} · "
      f"polars {pl.__version__} · numpy {np.__version__}")


Python 3.13.16 · duckdb 1.3.2 · polars 1.35.2 · numpy 2.1.3


## 1. Configuración

- **Semilla** reproducible para muestreos y desempates.
- **Muestreo opcional** (`USAR_MUESTREO`): la corrida interactiva de este
  notebook audita/entrena/evalúa una muestra aleatoria de usuarios, por
  velocidad. El método, tamaño, semilla y motivo quedan documentados en los
  metadatos de salida. Para cubrir el dataset **completo** basta
  `USAR_MUESTREO = False` (la ejecución completa es la opción por defecto de
  los scripts `src/` del proyecto).
- Todo lo generado se escribe bajo `nb_out/` (figuras, informe Markdown,
  `audit_results.json`, `nextcart_model.json`, `model_metrics.json`,
  manifiesto).


In [6]:
SEED = 42
MIN_SUPPORT = 5            # soporte mínimo (nº de cestas) para producto y par
TOP_K_RULES = 100          # máximo de reglas por antecedente
TOP_POPULAR = 1000         # tamaño de la lista de popularidad (fallback)
HISTORY_ITEMS = 50         # ítems recientes del historial usados como contexto
EVAL_KS = (1, 3, 5, 10)
LIFT_CAP = 5.0
POP_WEIGHT = 0.15          # mezcla de popularidad global en el score final
REPEAT_BOOST = 0.25        # boost para ítems ya comprados por el usuario
REJECTION_PENALTY = 0.2    # penalización por rechazo (usada en inferencia en vivo)
PAIR_SAMPLE_USERS = 30000  # usuarios usados en S4 (histograma de pares)

USAR_MUESTREO = False       # False = dataset completo (más lento, opción default del flujo)
N_SAMPLE_USERS = 30000

RUN_TS = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
OUT_DIR = Path("nb_out")
OUT_FIGS = OUT_DIR / "figures"
OUT_FIGS.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(SEED)
print(f"Timestamp de corrida: {RUN_TS} · salida en {OUT_DIR.resolve()}")


Timestamp de corrida: 2026-10-18T22:55:06Z · salida en /content/nb_out


## 2. Localización del dataset y verificación de archivos
localiza en disco el directorio con los seis
CSV (buscando `dataset/` o `data/` desde el directorio actual hacia arriba).
Si falta algo, imprime las instrucciones de corrección y aborta; nunca se
inventan datos para simular una auditoría.


In [1]:
 !curl -L -o dataset.zip https://www.kaggle.com/api/v1/datasets/download/yasserh/instacart-online-grocery-basket-analysis-dataset

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  197M  100  197M    0     0   107M      0  0:00:01  0:00:01 --:--:--  176M


In [4]:
!unzip dataset.zip -d data/

Archive:  dataset.zip
  inflating: data/aisles.csv         
  inflating: data/departments.csv    
  inflating: data/order_products__prior.csv  
  inflating: data/order_products__train.csv  
  inflating: data/orders.csv         
  inflating: data/products.csv       


In [7]:
DATASET_FILES = {
    "orders": "orders.csv",
    "order_products_prior": "order_products__prior.csv",
    "order_products_train": "order_products__train.csv",
    "products": "products.csv",
    "aisles": "aisles.csv",
    "departments": "departments.csv",
}

REQUIRED_COLUMNS = {
    "orders": ["order_id", "user_id", "eval_set", "order_number",
               "order_dow", "order_hour_of_day", "days_since_prior_order"],
    "order_products_prior": ["order_id", "product_id", "add_to_cart_order", "reordered"],
    "order_products_train": ["order_id", "product_id", "add_to_cart_order", "reordered"],
    "products": ["product_id", "product_name", "aisle_id", "department_id"],
    "aisles": ["aisle_id", "aisle"],
    "departments": ["department_id", "department"],
}


def find_data_dir() -> Path:
    "Busca dataset/ o data/ con orders.csv desde el cwd hacia la raíz."
    for base in (Path.cwd(), *Path.cwd().parents):
        for sub in ("dataset", "data"):
            if (base / sub / "orders.csv").is_file():
                return (base / sub).resolve()
    raise SystemExit(
        "ERROR: no se encontró el dataset Instacart.\n"
        "Descarga https://www.kaggle.com/datasets/yasserh/"
        "instacart-online-grocery-basket-analysis-dataset\n"
        "y coloca los 6 CSV en ./dataset/ o ./data/, luego re-ejecuta."
    )


DATA_DIR = find_data_dir()
PATHS = {k: DATA_DIR / v for k, v in DATASET_FILES.items()}
missing = [str(p) for p in PATHS.values() if not p.is_file()]
if missing:
    raise SystemExit(f"ERROR: faltan archivos del dataset: {missing}")

sizes = {k: p.stat().st_size for k, p in PATHS.items()}
print(f"Dataset localizado en: {DATA_DIR}")
for k, p in PATHS.items():
    print(f"  {k:22s} {p.name:28s} {sizes[k] / 1e6:9.1f} MB")


Dataset localizado en: /content/data
  orders                 orders.csv                       109.0 MB
  order_products_prior   order_products__prior.csv        577.6 MB
  order_products_train   order_products__train.csv         24.7 MB
  products               products.csv                       2.2 MB
  aisles                 aisles.csv                         0.0 MB
  departments            departments.csv                    0.0 MB


## 3. Conexión DuckDB y vistas del dataset

DuckDB lee los CSV **por consultas** (streaming desde disco) con un límite de
memoria configurable y volcado a un directorio *spill* en disco real (no en
`/tmp` si es tmpfs); así nunca se cargan en RAM los 32 M de filas de
`order_products__prior`. Si `USAR_MUESTREO`, las vistas se filtran a una
muestra aleatoria de usuarios; si no, se usan **todos**.


In [9]:
def connect_duckdb(memory_limit: str = "2GB") -> duckdb.DuckDBPyConnection:
    "Conexión en memoria con límite de RAM y spill a un directorio real."
    spill = Path("nb_spill").resolve()
    spill.mkdir(parents=True, exist_ok=True)
    con = duckdb.connect(database=":memory:")
    con.execute(f"SET memory_limit = '{memory_limit}'")
    con.execute(f"SET temp_directory = '{spill}'")
    con.execute("SET preserve_insertion_order = false")
    return con


def sample_user_ids(con: duckdb.DuckDBPyConnection, n: int) -> list[int]:
    "Muestra aleatoria (sin reemplazo) de user_ids del archivo orders."
    ids = sorted(int(r[0]) for r in con.execute(
        "SELECT user_id FROM read_csv_auto(?, header=true) GROUP BY 1",
        [str(PATHS["orders"])]).fetchall())
    return [int(u) for u in rng.choice(ids, size=min(n, len(ids)), replace=False)]


def build_views(con: duckdb.DuckDBPyConnection, user_ids: list[int] | None) -> None:
    "Crea las vistas del dataset, opcionalmente filtradas por usuario."
    where = ""
    if user_ids is not None:
        con.execute("CREATE OR REPLACE TEMP TABLE sample_users AS "
                    "SELECT unnest(?)::BIGINT AS user_id", [user_ids])
        where = "WHERE user_id IN (SELECT user_id FROM sample_users)"
    con.execute(
        "CREATE OR REPLACE VIEW orders_v AS "
        "SELECT order_id, user_id, eval_set, order_number, order_dow, "
        "TRY_CAST(order_hour_of_day AS INTEGER) AS order_hour_of_day, "
        "days_since_prior_order "
        f"FROM read_csv_auto('{PATHS['orders']}', header=true) {where}")
    con.execute(
        "CREATE OR REPLACE VIEW ops_prior_v AS "
        f"SELECT p.order_id, p.product_id, p.add_to_cart_order, p.reordered "
        f"FROM read_csv_auto('{PATHS['order_products_prior']}', header=true) p "
        "JOIN orders_v o USING (order_id)")
    con.execute(
        "CREATE OR REPLACE VIEW ops_train_v AS "
        "SELECT p.order_id, p.product_id, p.add_to_cart_order, p.reordered "
        f"FROM read_csv_auto('{PATHS['order_products_train']}', header=true) p "
        "JOIN orders_v o USING (order_id)")
    con.execute(f"CREATE OR REPLACE VIEW products_v AS "
                f"SELECT * FROM read_csv_auto('{PATHS['products']}', header=true)")
    con.execute(f"CREATE OR REPLACE VIEW aisles_v AS "
                f"SELECT * FROM read_csv_auto('{PATHS['aisles']}', header=true)")
    con.execute(f"CREATE OR REPLACE VIEW departments_v AS "
                f"SELECT * FROM read_csv_auto('{PATHS['departments']}', header=true)")


con = connect_duckdb("10GB")
SAMPLE_IDS = sample_user_ids(con, N_SAMPLE_USERS) if USAR_MUESTREO else None
build_views(con, SAMPLE_IDS)

sampling_meta = {
    "used": SAMPLE_IDS is not None,
    "method": "muestreo aleatorio de user_id (sin reemplazo)" if SAMPLE_IDS else None,
    "seed": SEED,
    "size": len(SAMPLE_IDS) if SAMPLE_IDS else None,
    "reason": "corrida interactiva del notebook; los scripts src/ usan el dataset completo",
    "description": (f"{len(SAMPLE_IDS):,} usuarios de 206,209 auditados"
                    "(~{:.1f}%)".format(100 * len(SAMPLE_IDS) / 206209)
                    if SAMPLE_IDS else "Sin muestreo: dataset completo."),
}
print(json.dumps(sampling_meta, indent=2, ensure_ascii=False))


{
  "used": false,
  "method": null,
  "seed": 42,
  "size": null,
  "reason": "corrida interactiva del notebook; los scripts src/ usan el dataset completo",
  "description": "Sin muestreo: dataset completo."
}


## 4. Supuesto S1 — Integridad del esquema

Verifica, con los CSV directamente (no con vistas, para que el chequeo sea
sobre los archivos reales):

- columnas requeridas en los seis archivos,
- claves primarias sin duplicados,
- huérfanos de referencia (orden, producto, pasillo, departamento),
- nulos de `days_since_prior_order` solo en la primera compra (esperado).


In [10]:
def check_s1(con: duckdb.DuckDBPyConnection) -> dict[str, Any]:
    "Integridad de esquema, unicidad, referencias y nulos."
    metrics: dict[str, Any] = {"columnas": {}, "unicidad": {}, "referencia": {},
                               "nulos": {}}
    ok = True
    for key, expected in REQUIRED_COLUMNS.items():
        cols = {r[0] for r in con.execute(
            "DESCRIBE SELECT * FROM read_csv_auto(?, header=true)",
            [str(PATHS[key])]).fetchall()}
        faltan = [c for c in expected if c not in cols]
        metrics["columnas"][key] = {"faltantes": faltan, "ok": not faltan}
        ok &= not faltan

    metrics["unicidad"]["dup_orders_order_id"] = int(con.execute(
        "SELECT count(*) - count(DISTINCT order_id) FROM "
        f"read_csv_auto('{PATHS['orders']}', header=true)").fetchone()[0])
    metrics["unicidad"]["dup_prior_order_product"] = int(con.execute(
        "SELECT count(*) - count(DISTINCT (order_id, product_id)) FROM "
        f"read_csv_auto('{PATHS['order_products_prior']}', header=true)"
    ).fetchone()[0])
    ok &= all(v == 0 for v in metrics["unicidad"].values())

    metr = [
        ("orphans_prior_order",
         "SELECT count(*) FROM read_csv_auto(?, header=true) p "
         "LEFT JOIN read_csv_auto(?, header=true) o USING(order_id) "
         "WHERE o.order_id IS NULL",
         [str(PATHS["order_products_prior"]), str(PATHS["orders"])]),
        ("orphans_prior_product",
         "SELECT count(*) FROM read_csv_auto(?, header=true) p "
         "LEFT JOIN products_v pr USING(product_id) WHERE pr.product_id IS NULL",
         [str(PATHS["order_products_prior"])]),
        ("orphans_train_product",
         "SELECT count(*) FROM read_csv_auto(?, header=true) p "
         "LEFT JOIN products_v pr USING(product_id) WHERE pr.product_id IS NULL",
         [str(PATHS["order_products_train"])]),
        ("orphans_product_aisle",
         "SELECT count(*) FROM products_v p LEFT JOIN aisles_v a USING(aisle_id) "
         "WHERE a.aisle_id IS NULL", None),
        ("orphans_product_department",
         "SELECT count(*) FROM products_v p LEFT JOIN departments_v d "
         "USING(department_id) WHERE d.department_id IS NULL", None),
    ]
    for name, sql, params in metr:
        metrics["referencia"][name] = int(
            con.execute(sql, params).fetchone()[0])
    ok &= all(v == 0 for v in metrics["referencia"].values())

    row = con.execute(
        "SELECT sum(CASE WHEN order_number = 1 AND days_since_prior_order "
        "IS NOT NULL THEN 1 ELSE 0 END), "
        "sum(CASE WHEN order_number > 1 AND days_since_prior_order IS NULL "
        "THEN 1 ELSE 0 END) "
        f"FROM read_csv_auto('{PATHS['orders']}', header=true)").fetchone()
    metrics["nulos"] = {
        "primera_orden_con_dias": int(row[0]),
        "no_primera_nulos": int(row[1]),
    }
    ok &= row[0] == 0 and row[1] == 0
    return {
        "id": "S1_schema_integrity",
        "status": "PASS" if ok else "FAIL",
        "metrics": metrics,
        "figures": [],
        "interpretation": (
            "Esquema completo, claves sin duplicados, sin huérfanos de referencia "
            "y nulos de days_since_prior_order solo en la primera compra "
            "(comportamiento esperado, no un error). Se pueden reconstruir cestas "
            "y secuencias sin imputación."),
        "limitations": [
            "No valida semántica del contenido (nombres de producto).",
            "order_hour_of_day llega como texto; se convierte con TRY_CAST.",
        ],
    }


t0 = time.perf_counter()
S1 = check_s1(con)
print(f"[S1] {S1['status']} ({time.perf_counter() - t0:.1f}s)")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[S1] PASS (38.2s)


## 5. Supuesto S2 — Historial secuencial

`order_number` debe numerar las órdenes de cada usuario **desde 1 y sin
huecos** (para que funcione como eje temporal de cestas). Además, cada usuario
debe tener exactamente una orden no-prior y, si es `train`, esa debe ser la
más reciente: así prior→train es una predicción de próxima cesta coherente.


In [11]:
def check_s2(con: duckdb.DuckDBPyConnection) -> tuple[dict[str, Any], list[Path]]:
    "Continuidad de order_number por usuario y estructura del split."
    gaps = int(con.execute(
        "SELECT count(*) FROM (SELECT user_id FROM orders_v GROUP BY user_id "
        "HAVING min(order_number) <> 1 OR max(order_number) <> count(*))"
    ).fetchone()[0])
    eval_counts = dict(con.execute(
        "SELECT eval_set, count(*) FROM orders_v GROUP BY 1").fetchall())
    multi_nonprior = int(con.execute(
        "SELECT count(*) FROM (SELECT user_id FROM orders_v WHERE eval_set <> 'prior' "
        "GROUP BY user_id HAVING count(*) > 1)").fetchone()[0])
    train_not_last = int(con.execute(
        "SELECT count(*) FROM (SELECT user_id, arg_max(eval_set, order_number) AS "
        "last_eval, max(CASE WHEN eval_set = 'train' THEN order_number END) AS "
        "train_on FROM orders_v GROUP BY user_id HAVING last_eval = 'train' AND "
        "train_on <> max(order_number))").fetchone()[0])
    users_total = int(con.execute(
        "SELECT count(DISTINCT user_id) FROM orders_v").fetchone()[0])
    days = con.execute(
        "SELECT avg(days_since_prior_order), "
        "quantile_cont(days_since_prior_order, 0.5), "
        "quantile_cont(days_since_prior_order, 0.9), "
        "max(days_since_prior_order) FROM orders_v "
        "WHERE days_since_prior_order IS NOT NULL").fetchone()

    counts = np.array([r[1] for r in con.execute(
        "SELECT user_id, count(*) FROM orders_v GROUP BY 1 ORDER BY 2").fetchall()])
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(counts, bins=np.arange(0, counts.max() + 2) - 0.5,
            color="#4C72B0", edgecolor="none")
    ax.set_title("Órdenes por usuario (S2)")
    ax.set_xlabel("número de órdenes")
    ax.set_ylabel("usuarios")
    ax.set_yscale("log")
    fig.tight_layout()
    f1 = OUT_FIGS / "nb_orders_per_user.png"
    fig.savefig(f1, dpi=130)
    plt.close(fig)

    bins = con.execute(
        "SELECT floor(days_since_prior_order / 5) * 5 AS b, count(*) AS n "
        "FROM orders_v WHERE days_since_prior_order IS NOT NULL "
        "GROUP BY 1 ORDER BY 1").fetchall()
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar([r[0] for r in bins], [r[1] for r in bins], width=4.5,
           color="#55A868", edgecolor="none")
    ax.set_title("Días desde la orden anterior (S2)")
    ax.set_xlabel("días")
    ax.set_ylabel("órdenes")
    fig.tight_layout()
    f2 = OUT_FIGS / "nb_days_since_prior.png"
    fig.savefig(f2, dpi=130)
    plt.close(fig)

    reorder_rate = float(con.execute(
        "SELECT avg(reordered) FROM ops_prior_v").fetchone()[0])
    ok = gaps == 0 and multi_nonprior == 0 and train_not_last == 0
    result = {
        "id": "S2_sequential_history",
        "status": "PASS" if ok else "FAIL",
        "metrics": {
            "usuarios_con_huecos_en_order_number": gaps,
            "usuarios_con_varias_ordenes_no_prior": multi_nonprior,
            "train_users_con_train_no_ultima": train_not_last,
            "eval_set_counts": {k: int(v) for k, v in eval_counts.items()},
            "dias_desde_orden_previa": {"media": float(days[0]),
                                        "mediana": float(days[1]),
                                        "p90": float(days[2]),
                                        "max": float(days[3])},
            "reorder_flag_rate_prior": reorder_rate,
            "usuarios_cubiertos": users_total,
        },
        "figures": [str(f1.relative_to(OUT_DIR)), str(f2.relative_to(OUT_DIR))],
        "interpretation": (
            "Las secuencias de compra de cada usuario son continuas (order_number "
            "sin huecos) y el split oficial marca una única orden train, la más "
            "reciente: la predicción de próxima cesta es coherente temporalmente. "
            f"El {100 * reorder_rate:.1f}% de los ítems prior está marcado como "
            "reordenado: repeat/explore es una partición real del problema."),
        "limitations": [
            "No hay timestamps absolutos: solo order_number, dow/hora y "
            "days_since_prior_order; la recencia solo puede ser relativa.",
        ],
    }
    return result, [f1, f2]


t0 = time.perf_counter()
S2, figs2 = check_s2(con)
print(f"[S2] {S2['status']} ({time.perf_counter() - t0:.1f}s)")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[S2] PASS (19.0s)


## 6. Supuesto S3 — Densidad de cestas y cola larga

Para que las reglas de coocurrencia funcionen: (a) casi todas las cestas deben
tener ≥ 2 productos, (b) la cesta media debe ser razonable (el conteo de pares
es cuadrático por cesta) y (c) el catálogo tiene **cola larga** — miles de
productos raros — lo que obliga a combinar reglas (cabeza asociativa) con
popularidad global (cold start y cola).


In [12]:
def check_s3(con: duckdb.DuckDBPyConnection) -> tuple[dict[str, Any], list[Path]]:
    "Densidad de cestas y estructura long-tail del catálogo."
    sizes = con.execute(
        "SELECT count(*), avg(c), quantile_cont(c, 0.5), quantile_cont(c, 0.9), "
        "quantile_cont(c, 0.99), max(c), sum(CASE WHEN c >= 2 THEN 1 ELSE 0 END) "
        "FROM (SELECT order_id, count(*) AS c FROM ops_prior_v GROUP BY 1)"
    ).fetchone()
    n_baskets = int(sizes[0])
    catalog = int(con.execute(
        "SELECT count(DISTINCT product_id) FROM products_v").fetchone()[0])
    seen = int(con.execute(
        "SELECT count(DISTINCT product_id) FROM ops_prior_v").fetchone()[0])

    bins = con.execute(
        "SELECT least(c, 60) AS b, count(*) AS n FROM "
        "(SELECT order_id, count(*) AS c FROM ops_prior_v GROUP BY 1) "
        "GROUP BY 1 ORDER BY 1").fetchall()
    freqs = np.array(sorted(
        (r[0] for r in con.execute(
            "SELECT count(*) FROM ops_prior_v GROUP BY product_id").fetchall()),
        reverse=True))
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].bar([r[0] for r in bins], [r[1] for r in bins],
                color="#4C72B0", edgecolor="none")
    axes[0].axvline(float(sizes[1]), color="red", ls="--",
                    label=f"media={float(sizes[1]):.1f}")
    axes[0].set_title("Tamaño de cestas prior (S3)")
    axes[0].legend()
    axes[1].loglog(np.arange(1, len(freqs) + 1), freqs, color="#4C72B0")
    axes[1].set_title("Cola larga: frecuencia de productos")
    fig.tight_layout()
    f1 = OUT_FIGS / "nb_basket_size_longtail.png"
    fig.savefig(f1, dpi=130)
    plt.close(fig)

    share_ge2 = int(sizes[6]) / n_baskets
    ok = share_ge2 > 0.9 and seen / catalog > 0.9
    result = {
        "id": "S3_basket_density",
        "status": "PASS" if ok else "WARN",
        "metrics": {
            "cestas_prior": n_baskets,
            "tamano_medio": float(sizes[1]),
            "mediana": float(sizes[2]),
            "p90": float(sizes[3]),
            "p99": float(sizes[4]),
            "maximo": int(sizes[5]),
            "share_cestas_con_2_o_mas_productos": share_ge2,
            "catalogo_distinto": catalog,
            "productos_con_prior": seen,
            "cobertura_catalogo_en_prior": seen / catalog,
        },
        "figures": [str(f1.relative_to(OUT_DIR))],
        "interpretation": (
            f"El {100 * share_ge2:.1f}% de las cestas prior tiene ≥ 2 productos "
            "(condición necesaria para coocurrencia) y la cesta media es de "
            f"{float(sizes[1]):.1f} productos: contar pares dentro de la cesta es "
            "barato. La cola larga del catálogo implica que el agente necesita "
            "un fallback de popularidad para cold start y productos sin reglas."),
        "limitations": [
            "Sin stock real: una cesta puede reflejar sustituciones, no demanda ideal.",
            "Miles de productos de cola quedarán sin reglas con min_support=5.",
        ],
    }
    return result, [f1]


t0 = time.perf_counter()
S3, figs3 = check_s3(con)
print(f"[S3] {S3['status']} ({time.perf_counter() - t0:.1f}s)")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[S3] PASS (52.8s)


## 7. Supuesto S4 — Señal de coocurrencia y recompra

Dos señales clave para el agente:

1. **Recompra** (repeat items): fracción de ítems que el usuario ya había
   comprado antes (definición estricta por aparición previa en su secuencia).
2. **Coocurrencia**: pares `(i, j)` que comparten cestas con soporte
   suficiente para reglas de confianza con `min_support = 5`.

Por costo computacional, el conteo de pares usa una muestra de usuarios (lo
documenta la sección de métricas); el **entrenamiento del modelo sí usa todas
las cestas incluidas** en la corrida (muestra del notebook o dataset completo).


In [13]:
def check_s4(con: duckdb.DuckDBPyConnection, max_users: int) -> tuple[dict[str, Any], list[Path]]:
    "Señal de recompra y conteo de pares de coocurrencia (muestra)."
    reorder_rate = float(con.execute(
        "SELECT avg(reordered) FROM ops_prior_v").fetchone()[0])
    users = [r[0] for r in con.execute(
        f"SELECT user_id FROM (SELECT DISTINCT o.user_id FROM ops_prior_v p "
        f"JOIN orders_v o USING(order_id) ORDER BY hash(o.user_id || '{SEED}') "
        f"LIMIT {max_users})").fetchall()]

    repeat_share = float(con.execute(
        "WITH sec AS (SELECT p.order_id, p.product_id, row_number() OVER "
        "(PARTITION BY o.user_id, p.product_id ORDER BY o.order_number) AS rn "
        "FROM ops_prior_v p JOIN orders_v o USING(order_id)), "
        "per_order AS (SELECT order_id, max(CASE WHEN rn > 1 THEN 1 ELSE 0 END) "
        "AS seen FROM sec GROUP BY order_id) "
        "SELECT avg(seen::DOUBLE) FROM per_order"
    ).fetchone()[0] if users else 0.0)

    con.execute("CREATE OR REPLACE TEMP TABLE s4_users AS "
                "SELECT unnest(?)::BIGINT AS user_id", [users])
    con.execute(
        "CREATE OR REPLACE TEMP TABLE s4_pairs AS "
        "SELECT a.product_id AS i, b.product_id AS j, count(*) AS c "
        "FROM ops_prior_v a "
        "JOIN ops_prior_v b ON a.order_id = b.order_id AND a.product_id < b.product_id "
        "WHERE a.order_id IN (SELECT order_id FROM orders_v "
        "WHERE user_id IN (SELECT user_id FROM s4_users)) GROUP BY 1, 2")
    stats = con.execute(
        "SELECT count(*), avg(c), max(c), sum(CASE WHEN c >= 2 THEN 1 ELSE 0 END), "
        "sum(CASE WHEN c >= 5 THEN 1 ELSE 0 END) FROM s4_pairs").fetchone()

    n_pairs = int(stats[0])
    cs = np.array([r[0] for r in con.execute(
        "SELECT least(c, 50) FROM s4_pairs").fetchall()], dtype=float)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(cs, bins=np.arange(0.5, 52, 1), color="#8172B3", edgecolor="none")
    ax.set_yscale("log")
    ax.set_title(f"Soporte de pares de coocurrencia (muestra de {len(users):,} usuarios)")
    ax.set_xlabel("co-ocurrencias por par (recortado a 50)")
    ax.set_ylabel("pares")
    fig.tight_layout()
    f1 = OUT_FIGS / "nb_pairs_support_hist.png"
    fig.savefig(f1, dpi=130)
    plt.close(fig)

    share_ge5 = (int(stats[4]) / n_pairs) if n_pairs else 0.0
    ok = n_pairs > 1000 and share_ge5 > 0.001
    result = {
        "id": "S4_complementarity_signal",
        "status": "PASS" if ok else "WARN",
        "metrics": {
            "reorder_flag_rate_prior": reorder_rate,
            "repeat_share_estrito_en_cestas": repeat_share,
            "muestra_pares": {
                "usuarios": len(users),
                "pares_distintos": n_pairs,
                "soporte_medio": float(stats[1]) if stats[1] is not None else None,
                "soporte_max": int(stats[2]) if stats[2] is not None else None,
                "share_pares_soporte_ge_5": share_ge5,
            },
        },
        "figures": [str(f1.relative_to(OUT_DIR))],
        "interpretation": (
            f"El {100 * reorder_rate:.1f}% de los ítems prior está marcado como "
            f"reordenado y el {100 * repeat_share:.1f}% de los ítems de cesta son "
            "repetidos en sentido estricto. En la muestra hay "
            f"{n_pairs:,} pares distintos y el {100 * share_ge5:.1f}% alcanza "
            "soporte ≥ 5: hay masa suficiente para reglas explicables con "
            "min_support = 5. El histograma muestra la típica cola larga de "
            "asociaciones."),
        "limitations": [
            "El conteo de pares de S4 es por muestra (solo diagnóstico); el "
            "entrenamiento usa todas las cestas de la corrida.",
            "Coocurrencia no implica causalidad ni complementariedad comercial "
            "deliberada; solo asociación histórica.",
        ],
    }
    return result, [f1]


t0 = time.perf_counter()
S4, figs4 = check_s4(con, PAIR_SAMPLE_USERS)
print(f"[S4] {S4['status']} ({time.perf_counter() - t0:.1f}s)")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[S4] PASS (105.8s)


## 8. Supuesto S5 — Evaluación sin fuga de información (leakage)

El split oficial solo es válido si:

1. los `order_id` de prior y train no se solapan (disjuntos);
2. la orden train de cada usuario es la más reciente de su secuencia;
3. cada usuario train tiene historial prior;
4. ninguna orden prior ocurre en la posición de la orden train del usuario.

Los usuarios `test` no tienen ground truth público y quedan fuera de la
evaluación supervisada.


In [14]:
def check_s5(con: duckdb.DuckDBPyConnection) -> tuple[dict[str, Any], list[Path]]:
    "Verificaciones anti-leakage del split oficial prior→train."
    overlap = int(con.execute(
        "SELECT count(*) FROM (SELECT order_id FROM ops_prior_v INTERSECT "
        "SELECT order_id FROM ops_train_v)").fetchone()[0])
    train_not_last = int(con.execute(
        "SELECT count(*) FROM (SELECT user_id FROM orders_v GROUP BY user_id "
        "HAVING max(CASE WHEN eval_set = 'train' THEN order_number END) IS NOT NULL "
        "AND max(CASE WHEN eval_set = 'train' THEN order_number END) "
        "<> max(order_number))").fetchone()[0])
    no_history = int(con.execute(
        "SELECT count(*) FROM (SELECT user_id FROM orders_v GROUP BY 1 "
        "HAVING sum(CASE WHEN eval_set = 'train' THEN 1 ELSE 0 END) > 0 "
        "AND sum(CASE WHEN eval_set = 'prior' THEN 1 ELSE 0 END) = 0)"
    ).fetchone()[0])
    prior_after_train = int(con.execute(
        "SELECT count(*) FROM orders_v t WHERE t.eval_set = 'train' "
        "AND EXISTS (SELECT 1 FROM orders_v p WHERE p.user_id = t.user_id "
        "AND p.eval_set = 'prior' AND p.order_number >= t.order_number)"
    ).fetchone()[0])
    train_users = int(con.execute(
        "SELECT count(DISTINCT user_id) FROM orders_v WHERE eval_set = 'train'"
    ).fetchone()[0])
    test_users = int(con.execute(
        "SELECT count(DISTINCT user_id) FROM orders_v WHERE eval_set = 'test'"
    ).fetchone()[0])

    checks = {
        "prior_y_train_disjuntos": overlap == 0,
        "train_es_ultima_orden_del_usuario": train_not_last == 0,
        "todo_usuario_train_tiene_historial_prior": no_history == 0,
        "ninguna_orden_prior_despues_de_train": prior_after_train == 0,
        "test_sin_label_publico": test_users > 0,
    }
    fig, ax = plt.subplots(figsize=(7, 3.2))
    ax.barh(list(checks), [int(bool(v)) for v in checks.values()],
            color=["#55A868" if v else "#C44E52" for v in checks.values()])
    ax.set_xlim(0, 1.2)
    ax.set_xticks([0, 1])
    ax.set_title("S5: verificaciones anti-leakage (1 = OK)")
    fig.tight_layout()
    f1 = OUT_FIGS / "nb_s5_leakage_checks.png"
    fig.savefig(f1, dpi=130)
    plt.close(fig)

    ok = all(checks.values())
    result = {
        "id": "S5_leakage_free_evaluation",
        "status": "PASS" if ok else "FAIL",
        "metrics": {"checks": checks, "train_users": train_users,
                    "test_users": test_users, "overlap_order_id": overlap},
        "figures": [str(f1.relative_to(OUT_DIR))],
        "interpretation": (
            "Split usable sin fuga: prior y train disjuntos, la orden train es la "
            "más reciente y toda orden prior ocurre estrictamente antes. Las "
            "reglas se entrenan solo con prior y la cesta objetivo nunca entra "
            "como variable. Los usuarios test quedan excluidos (sin ground truth "
            "público)."),
        "limitations": [
            "El split valida predicción de cesta, no conversión comercial: "
            "Instacart no contiene impresiones, exposición ni aceptación.",
        ],
    }
    return result, [f1]


t0 = time.perf_counter()
S5, figs5 = check_s5(con)
print(f"[S5] {S5['status']} ({time.perf_counter() - t0:.1f}s)")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[S5] PASS (18.1s)


## 9. Supuestos de referencia
| Supuesto de referencia | ¿Válido para NEXTCART? | Tratamiento |
|---|---|---|
| Homocedasticidad (Levene/Bartlett entre grupos) | **Adaptación útil**: comparar varianza del tamaño de cesta entre departamentos. | **S7_reference_aisle_density** (no bloqueante). |
| Ausencia de outliers (Z-score/IQR/Mahalanobis) | **Sí aplica**: cestas gigantes y usuarios power dominan los conteos y la popularidad. | **S6_reference_outliers** (no bloqueante). |


In [15]:

def check_s6_outliers(con: duckdb.DuckDBPyConnection) -> dict[str, Any]:
    "Supuesto de referencia adaptado: outliers de cestas y usuarios power."
    b = con.execute(
        "SELECT quantile_cont(c, 0.99), max(c), "
        "sum(CASE WHEN c >= 50 THEN c ELSE 0 END)::DOUBLE / sum(c) "
        "FROM (SELECT order_id, count(*) AS c FROM ops_prior_v GROUP BY 1)"
    ).fetchone()
    power = con.execute(
        "WITH per_user AS (SELECT o.user_id, count(*) AS n FROM ops_prior_v p "
        "JOIN orders_v o USING(order_id) GROUP BY 1), "
        "q AS (SELECT quantile_cont(n, 0.99) AS p99 FROM per_user) "
        "SELECT (SELECT p99 FROM q), max(n), avg(CASE WHEN n >= "
        "(SELECT p99 FROM q) THEN 1.0 ELSE 0.0 END), sum(CASE WHEN n >= "
        "(SELECT p99 FROM q) THEN n ELSE 0 END)::DOUBLE / sum(n) "
        "FROM per_user").fetchone()
    big_item_share = float(b[2] or 0.0)
    power_item_share = float(power[3] or 0.0)
    ok = big_item_share <= 0.20 and power_item_share <= 0.25
    return {
        "id": "S6_reference_outliers",
        "status": "PASS" if ok else "WARN",
        "metrics": {
            "cestas": {"p99_tamano": float(b[0]), "max": int(b[1]),
                       "share_items_en_cestas_de_50_o_mas": big_item_share},
            "usuarios": {"p99_ordenes_prior": float(power[0]), "max": int(power[1]),
                         "share_usuarios_power_p99": float(power[2]),
                         "share_items_de_usuarios_power": power_item_share},
        },
        "figures": [],
        "interpretation": (
            "Adaptación del supuesto de outliers de la referencia: las cestas "
            f"gigantes (≥ 50 ítems) aportan el {100 * big_item_share:.1f}% de las "
            f"filas y los usuarios power (top ~1% por nº de órdenes) el "
            f"{100 * power_item_share:.1f}%. No invalidan el modelo, pero "
            "distorsionan los conteos de coocurrencia y la lista de popularidad; "
            "si dominan, conviene limitar (cap) el tamaño de cesta del conteo o "
            "reponderar usuarios durante el entrenamiento."),
        "limitations": [
            "Supuesto adaptado de la referencia (imágenes): aquí se miden shares "
            "de filas, no Z-score/IQR/Mahalanobis.",
            "No bloquea: criterio informativo (WARN tolerable).",
        ],
    }


t0 = time.perf_counter()
S6 = check_s6_outliers(con)
print(f"[S6] {S6['status']} ({time.perf_counter() - t0:.1f}s)")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[S6] PASS (36.4s)


In [16]:

def check_s7_aisle_density(con: duckdb.DuckDBPyConnection) -> dict[str, Any]:
    "Supuesto de referencia adaptado: varianza de densidad entre departamentos."
    con.execute(
        "CREATE OR REPLACE TEMP TABLE basket_dept AS SELECT order_id, "
        "department_id FROM (SELECT p.order_id, pr.department_id, "
        "row_number() OVER (PARTITION BY p.order_id ORDER BY count(*) DESC) AS rk "
        "FROM ops_prior_v p JOIN products_v pr USING(product_id) "
        "GROUP BY 1, 2) WHERE rk = 1")
    rows = con.execute(
        "WITH sizes AS (SELECT order_id, count(*) AS c FROM ops_prior_v GROUP BY 1) "
        "SELECT bd.department_id, count(*) AS cestas, avg(s.c) AS media, "
        "var_pop(s.c) AS varianza FROM basket_dept bd JOIN sizes s USING(order_id) "
        "GROUP BY 1 HAVING count(*) >= 1000 ORDER BY cestas DESC").fetchall()
    means = [float(r[2]) for r in rows]
    ratio = (max(means) / min(means)) if means and min(means) > 0 else 0.0
    ok = ratio <= 3.0
    return {
        "id": "S7_reference_aisle_density",
        "status": "PASS" if ok else "WARN",
        "metrics": {
            "departamentos_con_1000_cestas": len(rows),
            "media_cesta_por_departamento": [
                {"department_id": int(r[0]), "cestas": int(r[1]),
                 "media_tamano": round(float(r[2]), 2),
                 "varianza": round(float(r[3]), 2)} for r in rows[:8]],
            "ratio_media_max_min": round(ratio, 3),
        },
        "figures": [],
        "interpretation": (
            "Adaptación del supuesto de homocedasticidad de la referencia: la "
            "media de tamaño de cesta por departamento (dominante en la cesta) "
            f"varía hasta {ratio:.2f}x entre el extremo mayor y el menor. No rompe "
            "el modelo, pero indica que las reglas pueden sesgar hacia "
            "departamentos de cesta grande; si el ratio fuese muy alto conviene "
            "reglas por departamento o normalización."),
        "limitations": [
            "Departamento dominante por conteo de ítems: aproximación, no "
            "etiquetado real de la cesta.",
            "Verificación no bloqueante (informativa).",
        ],
    }


t0 = time.perf_counter()
S7 = check_s7_aisle_density(con)
print(f"[S7] {S7['status']} ({time.perf_counter() - t0:.1f}s)")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[S7] WARN (44.1s)


## 10. Decisión de la auditoría y persistencia

Se consolidan los cinco supuestos (con métricas, figuras, interpretación y
limitaciones) y se decide la aptitud: un FAIL en **S1, S2 o S5** hace el
dataset *no apto*; WARN no bloquea pero queda registrado. Se persiste
`nb_out/audit_results.json` y el informe `nb_out/dataset_audit.md`.


In [17]:
ASSUMPTIONS = {r["id"]: {k: v for k, v in r.items() if k != "id"}
               for r in (S1, S2, S3, S4, S5, S6, S7)}
statuses = {k: v["status"] for k, v in ASSUMPTIONS.items()}
critical_ok = all(statuses[k] != "FAIL"
                  for k in ("S1_schema_integrity", "S2_sequential_history",
                            "S5_leakage_free_evaluation"))
decision = {
    "dataset_usable": critical_ok,
    "status": "usable_with_restrictions" if critical_ok else "not_usable",
    "reason": (
        "Se puede entrenar y evaluar offline un recomendador de próxima cesta "
        "sobre el split oficial prior/train sin fuga de información. Restricción: "
        "el dataset no tiene inventario real, precios, exposición del widget ni "
        "aceptación; el objetivo de ≥ 12 % de aceptación solo puede medirse con "
        "un A/B real de dos semanas en producción."),
    "required_actions": [
        "Entrenar solo con órdenes prior (nunca con la cesta objetivo).",
        "Aplicar filtros de inventario/rechazo en inferencia con datos de la tienda.",
        "Diseñar un A/B de dos semanas para medir la tasa de aceptación.",
    ],
    "risks": [
        "La coocurrencia refleja hábitos de usuarios de Instacart 2017; puede no transferirse a otra tienda.",
        "La evaluación offline no mide satisfacción ni aceptación de usuarios.",
    ],
}

audit_results = {
    "metadata": {
        "dataset": "instacart-online-grocery-basket-analysis-dataset",
        "executed_at_utc": RUN_TS,
        "python_version": platform.python_version(),
        "seed": SEED,
        "platform": f"{platform.system()} {platform.machine()} (equipo local, notebook)",
        "data_path": str(DATA_DIR),
        "sampling": sampling_meta,
    },
    "files": {k: {"size_bytes": sizes[k], "columns": REQUIRED_COLUMNS[k]}
              for k in DATASET_FILES},
    "assumptions": ASSUMPTIONS,
    "decision": decision,
}
(OUT_DIR / "audit_results.json").write_text(
    json.dumps(audit_results, ensure_ascii=False, indent=2), encoding="utf-8")

DESCRIPTIONS = {
    "S1_schema_integrity": "Integridad de esquema, claves y referencias",
    "S2_sequential_history": "Historial secuencial reconstruible por usuario",
    "S3_basket_density": "Densidad de cestas y cola larga del catálogo",
    "S4_complementarity_signal": "Señal de coocurrencia y recompra",
    "S5_leakage_free_evaluation": "Evaluación sin fuga de información (leakage)",
    "S6_reference_outliers": "Outliers de cestas/usuarios (adaptado de la referencia)",
    "S7_reference_aisle_density": "Varianza de densidad entre departamentos (adaptado)",
}
table_rows = "\n".join(
    f"| {sid} | {DESCRIPTIONS[sid]} | **{ASSUMPTIONS[sid]['status']}** |"
    for sid in DESCRIPTIONS)

sec4 = "\n\n".join(
    f"### {sid} — {ASSUMPTIONS[sid]['status']}\n\n"
    + ASSUMPTIONS[sid]["interpretation"] + "\n\n**Limitaciones:**\n"
    + "\n".join(f"- {lim}" for lim in ASSUMPTIONS[sid]["limitations"])
    for sid in DESCRIPTIONS)

report = (
    "# Auditoría del dataset — NEXTCART (notebook independiente)\n\n"
    f"**Corrida:** {RUN_TS} · **Seed:** {SEED} · **Dataset:** `{DATA_DIR}`\n"
    f"· **Muestreo:** {sampling_meta['description']}\n\n"
    "## 1. Resumen de la decisión\n\n"
    f"- dataset_usable: **{decision['dataset_usable']}** (`{decision['status']}`)\n"
    f"- {decision['reason']}\n\n"
    "## 2. Descripción del dataset\n\n"
    "Cada `order_id` es una cesta y `order_number` ordena las compras de cada "
    "usuario. El split oficial marca por usuario una última orden `train` "
    "(ground truth público) o `test` (sin ground truth); todo lo anterior es "
    "`prior`.\n\n"
    "## 3. Tabla de supuestos\n\n"
    "| Supuesto | Descripción | Estado |\n"
    "|---|---|---|\n" + table_rows + "\n\n"
    "## 4. Interpretación por supuesto\n\n" + sec4 + "\n\n"
    "## 5. Riesgos de leakage y preprocesamiento recomendado\n\n"
    "- Entrenar únicamente con órdenes `prior`; la cesta objetivo nunca entra "
    "como variable.\n"
    "- Cesta por `order_id`; secuencia con `order_number`.\n"
    "- Fallback de popularidad para cold start y productos sin reglas.\n\n"
    "## 6. Conclusión\n\n"
    f"El dataset es {'**apto**' if decision['dataset_usable'] else '**no apto**'} "
    f"para entrenar y evaluar offline NEXTCART (`{decision['status']}`). El "
    "objetivo comercial de ≥ 12 % de aceptación **no** puede validarse con "
    "Instacart: exige un A/B real de dos semanas con grupo de control en "
    "producción.\n")
(OUT_DIR / "dataset_audit.md").write_text(report, encoding="utf-8")

print("Estados:", json.dumps(statuses, ensure_ascii=False))
print("Persistidos:", (OUT_DIR / "audit_results.json").resolve(), "y",
      (OUT_DIR / "dataset_audit.md").resolve())


Estados: {"S1_schema_integrity": "PASS", "S2_sequential_history": "PASS", "S3_basket_density": "PASS", "S4_complementarity_signal": "PASS", "S5_leakage_free_evaluation": "PASS", "S6_reference_outliers": "PASS", "S7_reference_aisle_density": "WARN"}
Persistidos: /content/nb_out/audit_results.json y /content/nb_out/dataset_audit.md


## 11. Entrenamiento — baseline explicable (sin LLM)

Modelo `cooccurrence_frequency`, entrenado **solo con órdenes prior**:

1. **Soporte** de cada producto (número de cestas donde aparece).
2. **Popularidad global con recencia relativa** (posición media de la orden en
   la secuencia del usuario) → fallback para cold start y cola larga.
3. **Pares de coocurrencia** dentro de cada cesta (solo productos con
   soporte ≥ `MIN_SUPPORT`) → `count` por par.
4. **Reglas** por antecedente `i`: `confidence = count(i,j)/count(i)` y
   `lift = confidence / (support(j)/n_baskets)`, top-K por antecedente.
5. La mezcla final (regla + popularidad + boost de recompra) y los filtros de
   inventario/rechazo se aplican en **inferencia/evaluación**, no en el artefacto.

La cesta objetivo (train) **nunca** entra al entrenamiento.


In [18]:
def build_tables_for_training(con: duckdb.DuckDBPyConnection) -> dict[str, Any]:
    "Soporte con MIN_SUPPORT y popularidad con recencia relativa."
    n_baskets = int(con.execute(
        "SELECT count(DISTINCT order_id) FROM ops_prior_v").fetchone()[0])
    if n_baskets == 0:
        raise SystemExit("No hay órdenes prior en la corrida: nada que entrenar.")
    con.execute("CREATE OR REPLACE TABLE sup AS "
                "SELECT product_id, count(*) AS support FROM ops_prior_v "
                f"GROUP BY 1 HAVING count(*) >= {MIN_SUPPORT}")
    n_eligible = int(con.execute("SELECT count(*) FROM sup").fetchone()[0])
    con.execute(
        "CREATE OR REPLACE TABLE pop AS "
        "SELECT p.product_id, count(*) AS cnt, "
        "avg(o.order_number::DOUBLE / u.n_orders) AS recency "
        "FROM ops_prior_v p "
        "JOIN orders_v o USING(order_id) "
        "JOIN (SELECT user_id, count(*) AS n_orders FROM orders_v "
        "WHERE eval_set = 'prior' GROUP BY 1) u USING(user_id) "
        "GROUP BY 1")
    return {"n_baskets": n_baskets, "n_productos_con_soporte": n_eligible}


def generate_pairs_and_rules(con: duckdb.DuckDBPyConnection) -> int:
    "Pares de coocurrencia (solo productos con soporte) y reglas top-K."
    con.execute(
        "CREATE OR REPLACE TABLE pairs AS "
        "SELECT a.product_id AS i, b.product_id AS j, count(*) AS c "
        "FROM ops_prior_v a "
        "JOIN ops_prior_v b ON a.order_id = b.order_id "
        "AND a.product_id < b.product_id "
        "WHERE a.product_id IN (SELECT product_id FROM sup) "
        "AND b.product_id IN (SELECT product_id FROM sup) "
        "GROUP BY 1, 2")
    con.execute(
        f"CREATE OR REPLACE TABLE rules AS "
        f"SELECT i, j, c, 1.0 * c / s_i.support AS confidence "
        f"FROM pairs p JOIN sup s_i ON s_i.product_id = p.i "
        f"QUALIFY row_number() OVER (PARTITION BY i ORDER BY c DESC, j) "
        f"<= {TOP_K_RULES}")
    return int(con.execute("SELECT count(*) FROM pairs").fetchone()[0])


t0 = time.perf_counter()
train_stats = build_tables_for_training(con)
print(f"[train] tablas base: {json.dumps(train_stats)} "
      f"({time.perf_counter() - t0:.1f}s)")

t0 = time.perf_counter()
n_pairs = generate_pairs_and_rules(con)
print(f"[train] pares generados: {n_pairs:,} ({time.perf_counter() - t0:.1f}s)")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[train] tablas base: {"n_baskets": 3214874, "n_productos_con_soporte": 47618} (43.1s)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[train] pares generados: 40,660,881 (135.5s)


### Ensamblado del artefacto del modelo

Función que construye el dict del modelo (formato del entregable: parámetros,
`popular_products`, `association_rules`, `catalog_metadata`) y función de
**validación pre-guardado** con criterios mínimos; si no los supera, se marca
`experimental` en lugar de `candidate` (producción).


In [19]:
MODEL_VERSION = "notebook-0.1.0-" + RUN_TS.replace(":", "")


def build_model(con: duckdb.DuckDBPyConnection,
                pairs_count: int, stats_base: dict[str, Any]) -> dict[str, Any]:
    "Arma el dict completo del modelo (formato del entregable)."
    rules_df = con.execute(
        "WITH nb AS (SELECT count(DISTINCT order_id) AS nb FROM ops_prior_v) "
        "SELECT r.i, r.j, r.c, r.confidence, "
        "r.confidence / (s_j.support::DOUBLE / (SELECT nb FROM nb)) AS lift "
        "FROM rules r JOIN sup s_j ON s_j.product_id = r.j "
        "QUALIFY row_number() OVER (PARTITION BY r.i "
        "ORDER BY r.confidence DESC, r.j) <= " + str(TOP_K_RULES)).pl()

    assoc: dict[int, list[list[Any]]] = {}
    for i, j, c, conf, lift in zip(rules_df["i"], rules_df["j"], rules_df["c"],
                                   rules_df["confidence"], rules_df["lift"]):
        assoc.setdefault(int(i), []).append(
            [int(j), int(c), float(conf), float(lift)])
    for lst in assoc.values():
        lst.sort(key=lambda r: (-r[2], -r[3]))

    pop_df = con.execute(
        "SELECT product_id, recency, 1.0 * cnt / (SELECT max(cnt) FROM pop) AS score "
        "FROM pop ORDER BY cnt DESC LIMIT ? ", [TOP_POPULAR]).pl()
    popular = [{"product_id": int(pid), "score": round(float(sc), 6),
                "recency_rel": round(float(rec), 4)}
               for pid, sc, rec in zip(pop_df["product_id"], pop_df["score"],
                                       pop_df["recency"])]

    catalog_metadata = {
        "products": [{"product_id": int(r[0]), "product_name": r[1],
                      "aisle_id": int(r[2]), "department_id": int(r[3])}
                     for r in con.execute(
            "SELECT product_id, product_name, aisle_id, department_id "
            "FROM products_v").fetchall()],
        "aisles": [{"aisle_id": int(r[0]), "aisle": r[1]} for r in con.execute(
            "SELECT aisle_id, aisle FROM aisles_v").fetchall()],
        "departments": [{"department_id": int(r[0]), "department": r[1]}
                        for r in con.execute(
            "SELECT department_id, department FROM departments_v").fetchall()],
    }

    return {
        "model_version": MODEL_VERSION,
        "trained_at_utc": RUN_TS,
        "algorithm": "cooccurrence_frequency",
        "parameters": {
            "seed": SEED, "min_support": MIN_SUPPORT, "top_k_rules": TOP_K_RULES,
            "top_popular": TOP_POPULAR, "pop_weight": POP_WEIGHT,
            "repeat_boost": REPEAT_BOOST, "rejection_penalty": REJECTION_PENALTY,
            "lift_cap": LIFT_CAP, "history_items_used": HISTORY_ITEMS,
        },
        "status": "candidate",
        "format_version": 1,
        "sampling": sampling_meta,
        "training_statistics": {
            **stats_base,
            "pairs_count": pairs_count,
            "rules_count": sum(len(v) for v in assoc.values()),
            "antecedentes_con_reglas": len(assoc),
            "popular_count": len(popular),
        },
        "popular_products": popular,
        "association_rules": {str(k): v for k, v in assoc.items()},
        "catalog_metadata": catalog_metadata,
    }


def validate_model(m: dict[str, Any]) -> list[str]:
    "Criterios mínimos pre-guardado. Lista vacía = pasa."
    problems: list[str] = []
    if m["training_statistics"]["n_baskets"] < 100:
        problems.append("menos de 100 cestas prior en la corrida")
    if m["training_statistics"]["antecedentes_con_reglas"] < 50:
        problems.append("muy pocos antecedentes con reglas (< 50)")
    if m["training_statistics"]["popular_count"] < 50:
        problems.append("lista de popularidad demasiado corta (< 50)")
    confs = [r[2] for rs in m["association_rules"].values() for r in rs]
    if not confs or max(confs) <= 0.05:
        problems.append("ninguna regla con confianza > 0.05")
    return problems


t0 = time.perf_counter()
model = build_model(con, n_pairs, train_stats)
problems = validate_model(model)
model["status"] = "candidate" if not problems else "experimental"
mn = model["training_statistics"]
print(f"Modelo ensamblado: {mn['antecedentes_con_reglas']} antecedentes con reglas, "
      f"{mn['rules_count']} reglas, {mn['popular_count']} populares "
      f"({time.perf_counter() - t0:.1f}s)")
print("Estado de validación:", model["status"]
      + ((" — " + "; ".join(problems)) if problems else ""))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Modelo ensamblado: 47494 antecedentes con reglas, 4007904 reglas, 1000 populares (23.0s)
Estado de validación: candidate


### Persistencia del artefacto

Se guarda `nb_out/nextcart_model.json` y su **manifiesto** con versión, fecha
y hash sha256 (como el entregable del proyecto). Cargar este archivo **no es
entrenar de nuevo**: las celdas siguientes solo lo leen.


In [20]:
art_path = OUT_DIR / "nextcart_model.json"
art_path.write_text(json.dumps(model, ensure_ascii=False), encoding="utf-8")

sha256 = hashlib.sha256(art_path.read_bytes()).hexdigest()
manifest = {
    "model_path": art_path.name,
    "model_version": MODEL_VERSION,
    "created_at_utc": RUN_TS,
    "source_dataset": "instacart-online-grocery-basket-analysis-dataset",
    "training_code_version": "notebook-independiente (sin dependencia de src/)",
    "audit_results_path": str((OUT_DIR / "audit_results.json").resolve()),
    "metrics_path": str((OUT_DIR / "model_metrics.json").resolve()),
    "sha256": sha256,
}
(OUT_DIR / "artifact_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Artefacto: {art_path.resolve()} "
      f"({art_path.stat().st_size / 1e6:.1f} MB) · sha256={sha256[:12]}…")


Artefacto: /content/nb_out/nextcart_model.json (212.6 MB) · sha256=1bb2025d4d2b…


In [21]:
# Inspección de las reglas aprendidas: ¿qué se compra con bananos (24852)?
catalog_metadata = model["catalog_metadata"]
names = {p["product_id"]: p["product_name"] for p in catalog_metadata["products"]}


def show_rules(product_id: int, n: int = 5) -> None:
    "Muestra las mejores reglas del antecedente con nombres del catálogo."
    rules = model["association_rules"].get(str(product_id), [])
    print(f"Reglas de {product_id} ({names.get(product_id, '?')}):")
    if not rules:
        print("  (sin reglas — sin soporte suficiente)")
        return
    for j, c, conf, lift in rules[:n]:
        print(f"  -> {j:>6} {names.get(j, '?')[:38]:38s} "
              f"c={c:>7,} conf={conf:.3f} lift={lift:.2f}")


show_rules(24852)
show_rules(13176)


Reglas de 24852 (Banana):
  ->  47766 Organic Avocado                        c= 53,395 conf=0.113 lift=2.05
  ->  47626 Large Lemon                            c= 40,880 conf=0.087 lift=1.82
  ->  28204 Organic Fuji Apple                     c= 33,943 conf=0.072 lift=2.58
  ->  49683 Cucumber Kirby                         c= 32,097 conf=0.068 lift=2.24
  ->  26209 Limes                                  c= 31,772 conf=0.067 lift=1.54
Reglas de 13176 (Bag of Organic Bananas):
  ->  47209 Organic Hass Avocado                   c= 62,341 conf=0.164 lift=2.47
  ->  21137 Organic Strawberries                   c= 61,628 conf=0.162 lift=1.97
  ->  21903 Organic Baby Spinach                   c= 50,372 conf=0.133 lift=1.76
  ->  27966 Organic Raspberries                    c= 40,503 conf=0.107 lift=2.50
  ->  27845 Organic Whole Milk                     c= 26,481 conf=0.070 lift=1.63


## 12. Evaluación offline

**Split:** el oficial Instacart (historial = órdenes `prior`; objetivo = la
orden `train`, la más reciente, verificada en S5). Se evalúan tres variantes:

1. `topfreq`: popularidad pura (baseline global).
2. `cooccur`: reglas de coocurrencia por confianza, sin mezcla de popularidad
   ni boost de recompra (sin filtros avanzados).
3. `final`: modelo completo (reglas + popularidad + boost de recompra).

Para `K = 1, 3, 5, 10` se reportan Precision@K, Recall@K, HitRate@K y MAP@K,
más cobertura del catálogo, composición repeat/explore de los aciertos y
latencia de inferencia (media y p95).


In [22]:
def load_eval_users(con: duckdb.DuckDBPyConnection,
                    max_users: int | None) -> list[int]:
    "Usuarios con orden train; submuestreo opcional (documentado)."
    ids = sorted(int(r[0]) for r in con.execute(
        "SELECT DISTINCT user_id FROM orders_v WHERE eval_set = 'train'"
    ).fetchall())
    if max_users is not None and max_users < len(ids):
        rng_eval = np.random.default_rng(SEED)
        ids = sorted(int(u) for u in rng_eval.choice(
            ids, size=max_users, replace=False))
    return ids


def fetch_histories_targets(user_ids: list[int]
                            ) -> tuple[dict[int, list[int]], dict[int, set[int]]]:
    "Historial por usuario (lista cronológica de ítems prior) y cesta objetivo."
    hist = con.execute(
        "SELECT o.user_id, p.product_id FROM ops_prior_v p JOIN orders_v o "
        "USING(order_id) WHERE o.user_id IN (SELECT unnest(?)::BIGINT) "
        "ORDER BY o.user_id, o.order_number", [user_ids]).pl()
    tgt = con.execute(
        "SELECT o.user_id, p.product_id FROM ops_train_v p JOIN orders_v o "
        "USING(order_id) WHERE o.user_id IN (SELECT unnest(?)::BIGINT)",
        [user_ids]).pl()
    g1 = hist.group_by("user_id", maintain_order=True).agg(pl.col("product_id"))
    g2 = tgt.group_by("user_id", maintain_order=True).agg(pl.col("product_id"))
    histories = {int(u): [int(i) for i in items]
                 for u, items in zip(g1["user_id"], g1["product_id"])}
    targets = {int(u): {int(i) for i in items}
               for u, items in zip(g2["user_id"], g2["product_id"])}
    keep = [u for u in histories if u in targets]
    return {u: histories[u] for u in keep}, {u: targets[u] for u in keep}


# Submuestreo de evaluación solo para la corrida interactiva (queda documentado
# en el JSON de salida); con USAR_MUESTREO=False se evalúan todos los usuarios.
MAX_EVAL_USERS = 20000 if USAR_MUESTREO else None
eval_user_ids = load_eval_users(con, MAX_EVAL_USERS)
histories, targets = fetch_histories_targets(eval_user_ids)
print(f"Usuarios evaluables: {len(histories):,} "
      f"(de {len(eval_user_ids):,} usuarios train)")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Usuarios evaluables: 131,209 (de 131,209 usuarios train)


In [23]:
RULES = {int(k): [(int(r[0]), float(r[2]), float(r[3])) for r in v]
         for k, v in model["association_rules"].items()}
POP_IDS = [p["product_id"] for p in model["popular_products"]]
POP_NORM = {p["product_id"]: p["score"] for p in model["popular_products"]}
CATALOG_SIZE = len(model["catalog_metadata"]["products"])


def rank_for_user(history: list[int], variant: str, k_max: int) -> list[int]:
    "Ranking de productos para el usuario usando solo su historial previo."
    if variant == "topfreq":
        return POP_IDS[:k_max]
    context = history[-HISTORY_ITEMS:]
    history_set = set(history)
    best: dict[int, float] = {}
    for pid in context:
        for cons, conf, lift in RULES.get(pid, ()):
            score = conf * (0.5 + 0.5 * min(lift, LIFT_CAP) / LIFT_CAP)
            if variant == "final":
                score += POP_WEIGHT * POP_NORM.get(cons, 0.0)
                if cons in history_set:
                    score += REPEAT_BOOST * POP_NORM.get(cons, 0.0)
            prev = best.get(cons)
            if prev is None or score > prev:
                best[cons] = score
    return sorted(best, key=lambda p: (-best[p], p))[:k_max]


def evaluate_variant(variant: str, ks: tuple[int, ...]) -> dict[str, Any]:
    "Agrega métricas @K, cobertura, repeat/explore y latencia por usuario."
    agg = {"n": 0, "lat": [], "rec_ids": set(),
           "repeat_hits": 0, "explore_hits": 0,
           "precision": {k: 0.0 for k in ks}, "recall": {k: 0.0 for k in ks},
           "hitrate": {k: 0.0 for k in ks}, "map": {k: 0.0 for k in ks}}
    k_max = max(ks)
    for uid, target in targets.items():
        history = histories[uid]
        t0 = time.perf_counter()
        ranked = rank_for_user(history, variant, k_max)
        agg["lat"].append((time.perf_counter() - t0) * 1000.0)
        agg["n"] += 1
        agg["rec_ids"].update(ranked)
        hist_seen = set(history)
        hits_all = {p for p in ranked if p in target}
        for pid in hits_all:
            agg["repeat_hits" if pid in hist_seen else "explore_hits"] += 1
        for k in ks:
            top = ranked[:k]
            hits = [p for p in top if p in target]
            agg["precision"][k] += len(hits) / k
            agg["recall"][k] += (len(hits) / len(target)) if target else 0.0
            agg["hitrate"][k] += 1.0 if hits else 0.0
            if hits:
                cum = 0
                ap = 0.0
                for r, pid in enumerate(top):
                    if pid in target:
                        cum += 1
                        ap += cum / (r + 1)
                agg["map"][k] += ap / min(len(target), k)
    n = max(agg["n"], 1)
    lat = sorted(agg["lat"]) or [0.0]
    total_hits = agg["repeat_hits"] + agg["explore_hits"]
    return {
        "n_users": agg["n"],
        "metrics_by_k": {str(k): {
            "precision_at_k": round(agg["precision"][k] / n, 6),
            "recall_at_k": round(agg["recall"][k] / n, 6),
            "hit_rate_at_k": round(agg["hitrate"][k] / n, 6),
            "map_at_k": round(agg["map"][k] / n, 6),
        } for k in ks},
        "cobertura_catalogo": {
            "productos_distintos_recomendados": len(agg["rec_ids"]),
            "tamano_catalogo": CATALOG_SIZE,
            "fraccion": round(len(agg["rec_ids"]) / CATALOG_SIZE, 6),
        },
        "repeat_explore": {
            "repeat_hits": int(agg["repeat_hits"]),
            "explore_hits": int(agg["explore_hits"]),
            "repeat_share": (round(agg["repeat_hits"] / total_hits, 6)
                             if total_hits else None),
        },
        "latencia_ms": {
            "media": round(float(np.mean(lat)), 4),
            "p95": round(float(lat[min(len(lat) - 1, int(0.95 * len(lat)))]), 4),
        },
    }


t_start = time.perf_counter()
eval_results = {v: evaluate_variant(v, EVAL_KS)
                for v in ("topfreq", "cooccur", "final")}
eval_wall = time.perf_counter() - t_start
print(f"Evaluación completada en {eval_wall:.1f}s · usuarios={len(histories):,}")


Evaluación completada en 1120.1s · usuarios=131,209


In [24]:
rows = [{"K": k,
         **{f"P@{k} {v}": eval_results[v]["metrics_by_k"][str(k)]["precision_at_k"]
            for v in eval_results},
         **{f"R@{k} {v}": eval_results[v]["metrics_by_k"][str(k)]["recall_at_k"]
            for v in eval_results},
         **{f"HR@{k} {v}": eval_results[v]["metrics_by_k"][str(k)]["hit_rate_at_k"]
            for v in eval_results},
         **{f"MAP@{k} {v}": eval_results[v]["metrics_by_k"][str(k)]["map_at_k"]
            for v in eval_results}}
        for k in EVAL_KS]
print(pl.DataFrame(rows))

colors = {"topfreq": "#C44E52", "cooccur": "#DD8452", "final": "#4C72B0"}
panels = [("precision_at_k", "Precision@K"), ("recall_at_k", "Recall@K"),
          ("hit_rate_at_k", "HitRate@K"), ("map_at_k", "MAP@K")]
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for ax, (metric, title) in zip(axes.flat, panels):
    for variant, color in colors.items():
        ys = [eval_results[variant]["metrics_by_k"][str(k)][metric] for k in EVAL_KS]
        ax.plot(EVAL_KS, ys, marker="o", label=variant, color=color)
    ax.set_title(title)
    ax.set_xticks(EVAL_KS)
    ax.grid(alpha=0.3)
axes[0, 0].legend(fontsize=8)
fig.suptitle("Evaluación offline NEXTCART (split oficial prior→train)")
fig.tight_layout()
fig_path = OUT_FIGS / "nb_model_metrics_at_k.png"
fig.savefig(fig_path, dpi=130)
plt.close(fig)


shape: (4, 49)
┌─────┬────────────┬────────────┬───────────┬───┬────────────┬────────────┬────────────┬───────────┐
│ K   ┆ P@1        ┆ P@1        ┆ P@1 final ┆ … ┆ HR@10      ┆ MAP@10     ┆ MAP@10     ┆ MAP@10    │
│ --- ┆ topfreq    ┆ cooccur    ┆ ---       ┆   ┆ final      ┆ topfreq    ┆ cooccur    ┆ final     │
│ i64 ┆ ---        ┆ ---        ┆ f64       ┆   ┆ ---        ┆ ---        ┆ ---        ┆ ---       │
│     ┆ f64        ┆ f64        ┆           ┆   ┆ f64        ┆ f64        ┆ f64        ┆ f64       │
╞═════╪════════════╪════════════╪═══════════╪═══╪════════════╪════════════╪════════════╪═══════════╡
│ 1   ┆ 0.142719   ┆ 0.127651   ┆ 0.240852  ┆ … ┆ null       ┆ null       ┆ null       ┆ null      │
│ 3   ┆ null       ┆ null       ┆ null      ┆ … ┆ null       ┆ null       ┆ null       ┆ null      │
│ 5   ┆ null       ┆ null       ┆ null      ┆ … ┆ null       ┆ null       ┆ null       ┆ null      │
│ 10  ┆ null       ┆ null       ┆ null      ┆ … ┆ 0.553339   ┆ 0.044008   ┆ 

In [25]:
metrics_payload = {
    "metadata": {
        "generated_at_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "seed": SEED,
        "model_version": MODEL_VERSION,
        "data_path": str(DATA_DIR),
        "sampling": {**sampling_meta, "max_eval_users": MAX_EVAL_USERS},
        "wall_seconds_eval": round(eval_wall, 1),
    },
    "methodology": {
        "split": "oficial Instacart prior→train (usuarios con eval_set='train')",
        "description": (
            "Historial = todas las órdenes prior del usuario; objetivo = su orden "
            "train (la más reciente). Las reglas del modelo se entrenaron solo "
            "con prior. Los usuarios 'test' se excluyen por no tener ground "
            "truth. Offline no hay carrito parcial, inventario real ni rechazos: "
            "los filtros del modelo final son inercios aquí y se prueban solo en "
            "inferencia/backend."),
        "usuarios_evaluados": len(histories),
        "cestas_evaluadas": len(targets),
        "eval_ks": list(EVAL_KS),
        "history_items_used": HISTORY_ITEMS,
    },
    "models": eval_results,
    "baselines_evaluados": ["topfreq", "cooccur", "final"],
    "figures": ["figures/nb_model_metrics_at_k.png"],
    "warnings": [
        "Precision@K/Recall@K/HitRate@K/MAP@K son métricas offline de predicción "
        "de cesta; NO son tasas de conversión ni de aceptación comercial.",
        "Instacart no contiene impresiones de recomendaciones, exposición, grupo "
        "control ni aceptación: el objetivo de ≥ 12 % de aceptación solo puede "
        "medirse con un A/B real de dos semanas en producción.",
        "Los filtros de inventario y rechazo no pueden ejecutarse offline con "
        "datos reales de la tienda.",
    ],
}
(OUT_DIR / "model_metrics.json").write_text(
    json.dumps(metrics_payload, ensure_ascii=False, indent=2), encoding="utf-8")
print("Métricas guardadas en", (OUT_DIR / "model_metrics.json").resolve())

for v, r in eval_results.items():
    print(f"{v:8s} HR@10={r['metrics_by_k']['10']['hit_rate_at_k']:.4f} "
          f"R@10={r['metrics_by_k']['10']['recall_at_k']:.4f} "
          f"latencia_media={r['latencia_ms']['media']:.2f} ms "
          f"repeat_share={r['repeat_explore']['repeat_share']}")


Métricas guardadas en /content/nb_out/model_metrics.json
topfreq  HR@10=0.4584 R@10=0.0698 latencia_media=0.00 ms repeat_share=0.819307
cooccur  HR@10=0.5081 R@10=0.0872 latencia_media=3.39 ms repeat_share=0.857029
final    HR@10=0.5533 R@10=0.1005 latencia_media=5.01 ms repeat_share=0.880615


## 13. Conclusión

- La auditoría **S1–S5** sobre el dataset local establece aptitud: el split
  oficial prior→train es secuencial, denso en coocurrencia y **sin fuga** —
  apto para entrenar y evaluar offline el baseline explicable de NEXTCART.
- El modelo `cooccurrence_frequency` guarda **reglas auditables** por
  antecedente (soporte, confianza, lift) además de un fallback de popularidad
  con recencia relativa para cold start y cola larga; el artefacto se valida
  ante criterios mínimos y se persiste con manifiesto y hash.
- La evaluación offline lo posiciona contra `topfreq` y `cooccur` con
  Precision/Recall/HitRate/MAP @K, cobertura, repeat/explore y latencia.
- Las **limitaciones de Instacart** (sin inventario, precio, posición del
  widget, impresiones ni aceptación) implican que los filtros de sesión y el
  objetivo comercial de ≥ 12 % solo se validan con un **A/B real** de dos
  semanas en producción.

Archivos generados por este notebook (todo bajo `nb_out/`, independiente del
paquete `src/` del proyecto):

| Archivo | Contenido |
|---|---|
| `nb_out/audit_results.json` | resultados S1–S5 con métricas + decisión |
| `nb_out/dataset_audit.md` | informe legible de auditoría |
| `nb_out/nextcart_model.json` | artefacto del modelo entrenado |
| `nb_out/artifact_manifest.json` | manifiesto con sha256 del artefacto |
| `nb_out/model_metrics.json` | métricas offline de evaluación |
| `nb_out/figures/*.png` | figuras de auditoría y métricas |


In [26]:
print("Archivos generados por el notebook:")
for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file():
        print(f"  {p}  ({p.stat().st_size:,} bytes)")


Archivos generados por el notebook:
  nb_out/artifact_manifest.json  (490 bytes)
  nb_out/audit_results.json  (11,667 bytes)
  nb_out/dataset_audit.md  (5,802 bytes)
  nb_out/figures/nb_basket_size_longtail.png  (51,046 bytes)
  nb_out/figures/nb_days_since_prior.png  (21,993 bytes)
  nb_out/figures/nb_model_metrics_at_k.png  (151,574 bytes)
  nb_out/figures/nb_orders_per_user.png  (21,947 bytes)
  nb_out/figures/nb_pairs_support_hist.png  (29,961 bytes)
  nb_out/figures/nb_s5_leakage_checks.png  (32,897 bytes)
  nb_out/model_metrics.json  (5,087 bytes)
  nb_out/nextcart_model.json  (212,634,493 bytes)
